# Projet Spark : Analyse et modélisation Big Data avec PySpark
**Dataset :** Online Retail (UCI Machine Learning Repository)

**Auteur:** Cédric MANELLI

## 1) Initialisation et chargement des données


In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import NumericType
from pyspark.sql.functions import count, sum, avg, countDistinct, col
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
import matplotlib.pyplot as plt
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator, RegressionEvaluator
from pyspark.ml.classification import GBTClassifier, LogisticRegression
import plotly.express as px
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ProjetSpark").getOrCreate()

In [2]:

spark = (
    SparkSession.builder
    .appName("ProjetSpark")
    .getOrCreate()
)

print("Spark OK", spark.version)

Spark OK 3.5.3


In [3]:
# Install gdown

import gdown
import pandas as pd
import numpy as np

# Google Drive file ID
file_id = "1t5xNS23kfpLKOu5aMxevcHdRyyZ6UWLd"
url = f"https://drive.google.com/uc?id={file_id}"

# Local filename
filename = "Online_Retail_CSV.csv"

# Download the file
data = gdown.download(url= url, output = filename, quiet=False)



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\MANEL\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip
Downloading...
From: https://drive.google.com/uc?id=1t5xNS23kfpLKOu5aMxevcHdRyyZ6UWLd
To: c:\Users\MANEL\Documents\data-analytics\spark\projet\projet\Online_Retail_CSV.csv
100%|██████████| 46.1M/46.1M [00:02<00:00, 20.9MB/s]


In [4]:
# importation de la base de données source
df = (spark.read
      .option("header", True)
      .option("inferSchema", True)
      .option("sep", ";")
      .csv(data))

# print du nombre de lignes, colonnes et les types des variables
print("Nb lignes :", df.count())
print("Nb colonnes :", len(df.columns))
print("dtypes:", df.dtypes[:8])

Nb lignes : 541909
Nb colonnes : 8
dtypes: [('InvoiceNo', 'string'), ('StockCode', 'string'), ('Description', 'string'), ('Quantity', 'int'), ('InvoiceDate', 'string'), ('UnitPrice', 'string'), ('CustomerID', 'int'), ('Country', 'string')]


## 2) Exploration et pré-traitement

In [5]:
df.show(5, truncate=False)
print(df.columns)

+---------+---------+-----------------------------------+--------+----------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate     |UnitPrice|CustomerID|Country       |
+---------+---------+-----------------------------------+--------+----------------+---------+----------+--------------+
|536365   |85123A   |WHITE HANGING HEART T-LIGHT HOLDER |6       |01/12/2010 08:26|2,55     |17850     |United Kingdom|
|536365   |71053    |WHITE METAL LANTERN                |6       |01/12/2010 08:26|3,39     |17850     |United Kingdom|
|536365   |84406B   |CREAM CUPID HEARTS COAT HANGER     |8       |01/12/2010 08:26|2,75     |17850     |United Kingdom|
|536365   |84029G   |KNITTED UNION FLAG HOT WATER BOTTLE|6       |01/12/2010 08:26|3,39     |17850     |United Kingdom|
|536365   |84029E   |RED WOOLLY HOTTIE WHITE HEART.     |6       |01/12/2010 08:26|3,39     |17850     |United Kingdom|
+---------+---------+-------------------

On voit qu'il y a présence d'erreurs de formats dans les données. UnitPrice est un string avec un séparateur décimal de convention européenne, la date est un string ...\

In [6]:
# conversion des variables
# remplacer le séparateur décimal avant de cnvertir en float (prix en convention européenne)
df_conv = df.withColumn(
    "UnitPrice",
    F.regexp_replace(F.col("UnitPrice"), ",", ".").cast("float")
)
# convertir la date
df_conv = df_conv.withColumn(
    "InvoiceDate",
    F.to_timestamp("InvoiceDate", "dd/MM/yyyy HH:mm")
)
df_conv.show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
only showing top 5 rows



In [7]:
numeric_cols = [f.name for f in df_conv.schema.fields if isinstance(f.dataType, NumericType)]
non_numeric_cols = [c for c in df_conv.columns if c not in numeric_cols]

print("numeric:", numeric_cols[:20])
print("non-numeric:", non_numeric_cols[:20])

numeric: ['Quantity', 'UnitPrice', 'CustomerID']
non-numeric: ['InvoiceNo', 'StockCode', 'Description', 'InvoiceDate', 'Country']


In [8]:
# tester les doublons
before = df_conv.count()
df_conv = df_conv.dropDuplicates()
after = df_conv.count()
print("Avant :", before, "| Après :", after, "| Supprimées :", before - after)
# remplacer les NA par None
df_conv = df_conv.replace("NA", None)

Avant : 541909 | Après : 536641 | Supprimées : 5268


In [9]:
# affichage des clients uniques du montant total et de la moyenne des transactions
df_conv.agg(
    count("*").alias("total"),
    countDistinct("CustomerID").alias("clients_uniques"),
    sum("UnitPrice").alias("montant_total"),
    avg("UnitPrice").alias("montant_moyen")
).show()

# affichage par client, du nbr de transaction, du montant total et du montant moyen des transactions
df_conv = df_conv.withColumn("Montant", col("Quantity") * col("UnitPrice"))
(df_conv.groupBy("CustomerID")
  .agg(
      count("*").alias("nbr_transactions"),
      sum("Montant").alias("montant_total"),
      avg("Montant").alias("montant_moyen")
      )
  .orderBy(col("montant_total").desc())
).show()

#affichage les montants et transactions par pays
(df_conv.groupBy("Country")
   .agg(
       count("*").alias("nb_transactions"),
       sum("UnitPrice").alias("total")
   )
   .orderBy(col("total").desc())
   .show())

# affichage données et schéma
df_conv.show(5)
df_conv.printSchema()

# afficher les statistiques descriptives
df_conv.describe().show()



+------+---------------+-----------------+-----------------+
| total|clients_uniques|    montant_total|    montant_moyen|
+------+---------------+-----------------+-----------------+
|536641|           4372|2486072.967881962|4.632655663436007|
+------+---------------+-----------------+-----------------+

+----------+----------------+------------------+------------------+
|CustomerID|nbr_transactions|     montant_total|     montant_moyen|
+----------+----------------+------------------+------------------+
|      NULL|          135037|1447487.5282588564|10.719191986336014|
|     14646|            2085|279489.01930066943| 134.0474912713043|
|     18102|             433|256438.48995232582| 592.2366973494823|
|     17450|             350| 187322.1704902649| 535.2062014007569|
|     14911|            5898|132458.72962367535|22.458245104048043|
|     12415|             778|123725.44990730286| 159.0301412690268|
|     14156|            1415|113214.58939158916| 80.01031052409128|
|     17511|  

Nous avons des quantités qui sont négatives qui correspond certainement à des retours et des prix négatifs aussi correspondant aux remboursements.

In [10]:
df_conv.select(
    F.sum((F.col("Quantity") == 0).cast("int")).alias("Quantity_eq_0"),
    F.sum((F.col("Quantity") < 0).cast("int")).alias("Quantity_neg"),
    F.sum((F.col("UnitPrice") == 0).cast("int")).alias("UnitPrice_eq_0"),
    F.sum((F.col("UnitPrice") < 0).cast("int")).alias("UnitPrice_neg")
).show()

+-------------+------------+--------------+-------------+
|Quantity_eq_0|Quantity_neg|UnitPrice_eq_0|UnitPrice_neg|
+-------------+------------+--------------+-------------+
|            0|       10587|          2510|            2|
+-------------+------------+--------------+-------------+



In [11]:
# on compte les valeurs 'null'
null_counts = df_conv.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
null_counts.show(truncate=False,vertical=True)

-RECORD 0-------------
 InvoiceNo   | 0      
 StockCode   | 0      
 Description | 1454   
 Quantity    | 0      
 InvoiceDate | 0      
 UnitPrice   | 0      
 CustomerID  | 135037 
 Country     | 0      



In [12]:
# la proportion des valeurs 'null'
n = df_conv.count()

df_conv.select([
    (F.sum(F.col(c).isNull().cast("int")) / F.lit(n)).alias(c)
    for c in df_conv.columns
]).show(truncate=False,vertical=True)

-RECORD 0----------------------------
 InvoiceNo   | 0.0                   
 StockCode   | 0.0                   
 Description | 0.0027094463524031894 
 Quantity    | 0.0                   
 InvoiceDate | 0.0                   
 UnitPrice   | 0.0                   
 CustomerID  | 0.2516337737891812    
 Country     | 0.0                   
 Montant     | 0.0                   



Ici nous avons ~25% de valeurs manquantes pour le ID client.
Les meilleurs méthodes de filtrages serait:
- de supprimer les clients dont l'ID n'est pas présent (on souhaite faire de la clusterisation clientèle)
- de conserver les retours clients + remboursements pour éviter des erreurs de CA et CA/ clients

In [13]:
# on supprime les valeurs CustomerID Null
df_clean = df_conv.filter(
    F.col("CustomerID").isNotNull()
)
before = df_conv.count()
after = df_clean.count()

print("Lignes supprimées :", before - after)
df_clean.show()

Lignes supprimées : 135037
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+---------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|  Montant|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+---------+
|   536446|    21587|COSY HOUR GIANT T...|       4|2010-12-01 12:15:00|     2.55|     15983|United Kingdom|     10.2|
|   536556|    35961|FOLKART ZINC HEAR...|      12|2010-12-01 14:38:00|     0.85|     17643|United Kingdom|10.200001|
|   536563|    21452| TOADSTOOL MONEY BOX|      10|2010-12-01 15:08:00|     2.95|     17760|United Kingdom|     29.5|
|   536575|    21232|STRAWBERRY CERAMI...|     144|2010-12-01 16:01:00|     1.25|     13777|United Kingdom|    180.0|
|   536588|    21777|RECIPE BOX WITH M...|       1|2010-12-01 16:49:00|     7.95|     17069|United Kingdom|     7.95|
|   536591|   90214M|"LETTER 

## 3) Segmentation client

1.	Création de variables RFM (Recency, Frequency, Monetary) :
  -	Recency : nombre de jours (ou secondes) depuis la dernière commande.
  -	Frequency : nombre total de commandes passées.
   -	Monetary : total dépensé par client.
2.	Regrouper ces variables RFM dans un DataFrame PySpark par client.
3.	Assembler et standardiser ces features (VectorAssembler + StandardScaler).
4.	Entraîner un des 3 algos non supervisé proposé dans les objectifs :

-	Déterminer le nombre optimal de clusters (méthode du coude ou indice silhouette ?).
-	Obtenir les centroïdes et assigner chaque client à un cluster.

5.	Analyser les segments :
-	Moyenne de recency, frequency, monetary par cluster.
-	Interpréter les groupes (clients fidèles, gros dépensiers, etc.).


In [14]:
# création de la variable R
max_date = df_clean.agg(F.max("InvoiceDate")).first()[0]
df_r = (df_clean
    .groupBy("CustomerID")
    .agg(
        F.datediff(
            F.lit(max_date),
            F.to_date(F.max("InvoiceDate"))
        ).alias("Recency_days")
    ).orderBy(col("CustomerID"))
)


In [15]:
# regroupement par numéro de facture.
regroup = (df_clean
    .groupBy("CustomerID", "InvoiceNo")
    .agg(
        sum("Montant").alias("montant_commande")
    )
    .orderBy(col("CustomerID"))
)

In [16]:
# regroupement des fatures par client et du montant total
df_mf = (regroup
    .groupBy("CustomerID")
    .agg(
        F.count("*").alias("nbr_total_commandes"),                 # nombre de factures
        F.sum("montant_commande").alias("montant_total"),    # somme des commandes
    )
    .orderBy(col("CustomerID"))
)


In [17]:
df_f = df_mf.select("CustomerID", "nbr_total_commandes")
df_f.show()
df_m = df_mf.select("CustomerID", "montant_total")
df_m.show()
df_r.show()

+----------+-------------------+
|CustomerID|nbr_total_commandes|
+----------+-------------------+
|     12346|                  2|
|     12347|                  7|
|     12348|                  4|
|     12349|                  1|
|     12350|                  1|
|     12352|                 11|
|     12353|                  1|
|     12354|                  1|
|     12355|                  1|
|     12356|                  3|
|     12357|                  1|
|     12358|                  2|
|     12359|                  6|
|     12360|                  3|
|     12361|                  1|
|     12362|                 13|
|     12363|                  2|
|     12364|                  4|
|     12365|                  3|
|     12367|                  1|
+----------+-------------------+
only showing top 20 rows

+----------+------------------+
|CustomerID|     montant_total|
+----------+------------------+
|     12346|               0.0|
|     12347| 4310.000016212463|
|     12348|1797.23999

In [18]:
df_fm = df_f.join(df_m, on="CustomerID", how="outer")
df_rfm = df_fm.join(df_r, on="CustomerID", how="outer").orderBy(col("CustomerID"))

df_rfm.show()
df_rfm.count()

+----------+-------------------+------------------+------------+
|CustomerID|nbr_total_commandes|     montant_total|Recency_days|
+----------+-------------------+------------------+------------+
|     12346|                  2|               0.0|         325|
|     12347|                  7| 4310.000016212463|           2|
|     12348|                  4|1797.2399997711182|          75|
|     12349|                  1|1757.5499963760376|          18|
|     12350|                  1|334.39999771118164|         310|
|     12352|                 11|1545.4099912643433|          36|
|     12353|                  1|              89.0|         204|
|     12354|                  1| 1079.400016784668|         232|
|     12355|                  1| 459.3999996185303|         214|
|     12356|                  3| 2811.429984331131|          22|
|     12357|                  1| 6207.669998168945|          33|
|     12358|                  2|1168.0599336624146|           1|
|     12359|             

4372

Maintenant rassemblons les vecteurs (colonnes) en features avec VectorAssembler. Puis standardisons les features avec StandardScaler.

In [19]:
# selection des colones features
features_columns = ['nbr_total_commandes', 'montant_total', 'Recency_days']

# assemblage des features dans une liste pour chaque client
assembler = VectorAssembler(inputCols=features_columns, outputCol='features')
# rassemblement avec le df
final_df = assembler.transform(df_rfm)

# show features
final_df.select("CustomerID", "features").show(truncate=False)

# normalisation des features
scaler = StandardScaler(inputCol='features', outputCol='scaledFeatures', withStd=True, withMean=False)
scaler_model = scaler.fit(final_df)
final_df_scaled = scaler_model.transform(final_df)

#show scaledfeatures
final_df_scaled.select("CustomerID", "scaledFeatures").show(truncate=False)


+----------+------------------------------+
|CustomerID|features                      |
+----------+------------------------------+
|12346     |[2.0,0.0,325.0]               |
|12347     |[7.0,4310.000016212463,2.0]   |
|12348     |[4.0,1797.2399997711182,75.0] |
|12349     |[1.0,1757.5499963760376,18.0] |
|12350     |[1.0,334.39999771118164,310.0]|
|12352     |[11.0,1545.4099912643433,36.0]|
|12353     |[1.0,89.0,204.0]              |
|12354     |[1.0,1079.400016784668,232.0] |
|12355     |[1.0,459.3999996185303,214.0] |
|12356     |[3.0,2811.429984331131,22.0]  |
|12357     |[1.0,6207.669998168945,33.0]  |
|12358     |[2.0,1168.0599336624146,1.0]  |
|12359     |[6.0,6182.979964256287,7.0]   |
|12360     |[3.0,2662.060012817383,52.0]  |
|12361     |[1.0,189.90000534057617,287.0]|
|12362     |[13.0,5154.580017566681,3.0]  |
|12363     |[2.0,552.0000133514404,109.0] |
|12364     |[4.0,1313.1000204086304,7.0]  |
|12365     |[3.0,320.69000720977783,291.0]|
|12367     |[1.0,168.90000438690

Clusterisation avec Kmeans + optimisation de l'hyperparamètre k

In [33]:
"""
kmeans = KMeans(featuresCol = 'scaledFeatures', k=3)
model = kmeans.fit(final_df_scaled)
"""

"\nkmeans = KMeans(featuresCol = 'scaledFeatures', k=3)\nmodel = kmeans.fit(final_df_scaled)\n"

In [34]:
# On utilise model.summary pour obtenir les performances du model
"""
print("WSSSE:", model.summary.trainingCost)
"""

'\nprint("WSSSE:", model.summary.trainingCost)\n'

In [35]:
# réalisation d'une boucle afin de réaliser plusieurs kmeans en modifiant l'hyperparamètre k afin d'obtenir le plus adéquat à nos données
"""
ks = list(range(2, 10))
silouhette = []
wssse_score = []

# evaluateur silouhette
evaluator = ClusteringEvaluator(
    featuresCol="scaledFeatures",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

# boucle de variation du k
for k in ks:
  kmeans = KMeans(featuresCol = 'scaledFeatures', k=k, seed = 12)
  model = kmeans.fit(final_df_scaled)
  predictions = model.transform(final_df_scaled)
  sil = evaluator.evaluate(predictions)
  wssse = model.summary.trainingCost
  silouhette.append((k, sil))
  wssse_score.append((k, wssse))
  print(f"k : {k}, wssse : {wssse}, sil : {sil}")
  """


'\nks = list(range(2, 10))\nsilouhette = []\nwssse_score = []\n\n# evaluateur silouhette\nevaluator = ClusteringEvaluator(\n    featuresCol="scaledFeatures",\n    metricName="silhouette",\n    distanceMeasure="squaredEuclidean"\n)\n\n# boucle de variation du k\nfor k in ks:\n  kmeans = KMeans(featuresCol = \'scaledFeatures\', k=k, seed = 12)\n  model = kmeans.fit(final_df_scaled)\n  predictions = model.transform(final_df_scaled)\n  sil = evaluator.evaluate(predictions)\n  wssse = model.summary.trainingCost\n  silouhette.append((k, sil))\n  wssse_score.append((k, wssse))\n  print(f"k : {k}, wssse : {wssse}, sil : {sil}")\n  '

In [36]:
# Affichage de la courbe et détermination du coude
"""
silouhette_pd = pd.DataFrame(silouhette, columns=["k", "sil"])
wssse_pd = pd.DataFrame(wssse_score, columns=["k", "WSSSE"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(wssse_pd['k'], wssse_pd['WSSSE'], marker="o")
ax1.set_xlabel("k (nombre de clusters)")
ax1.set_ylabel("WSSSE (trainingCost)")
ax1.set_title("coude méthode (KMeans)")
ax1.set_xticks(ks)

ax2.plot(silouhette_pd['k'], silouhette_pd['sil'], marker="o")
ax2.set_xlabel("k (nombre de clusters)")
ax2.set_ylabel("sil")
ax2.set_title("silouhette méthode (KMeans)")
ax2.set_xticks(ks)


plt.show()
"""

'\nsilouhette_pd = pd.DataFrame(silouhette, columns=["k", "sil"])\nwssse_pd = pd.DataFrame(wssse_score, columns=["k", "WSSSE"])\n\nfig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))\n\nax1.plot(wssse_pd[\'k\'], wssse_pd[\'WSSSE\'], marker="o")\nax1.set_xlabel("k (nombre de clusters)")\nax1.set_ylabel("WSSSE (trainingCost)")\nax1.set_title("coude méthode (KMeans)")\nax1.set_xticks(ks)\n\nax2.plot(silouhette_pd[\'k\'], silouhette_pd[\'sil\'], marker="o")\nax2.set_xlabel("k (nombre de clusters)")\nax2.set_ylabel("sil")\nax2.set_title("silouhette méthode (KMeans)")\nax2.set_xticks(ks)\n\n\nplt.show()\n'

Sur la méthode du coude, la rupture n'est pas net. Parcontre grace à la silouhette, on peut voir qu'au bout de 5 groupes, la courbe arrête sa croissance. On prendra pour la suite la valeur k=5.

In [20]:

kmeans = KMeans(featuresCol = 'scaledFeatures', k=5)
model = kmeans.fit(final_df_scaled)


In [21]:

#¥ récupération des clusters
centroids = model.clusterCenters()
#attribution des coordonnées des clusters
for i, c in enumerate(centroids):
  print(f"cluster {i+1} : {c}")


cluster 1 : [0.40691384 0.13981085 0.43687619]
cluster 2 : [9.78003401 6.23827013 0.04678164]
cluster 3 : [0.19306514 0.05500307 2.47135653]
cluster 4 : [ 5.88943654 25.76367072  0.08186787]
cluster 5 : [2.14222518 0.87207347 0.13376713]


In [22]:
# répartitions des ID clients dans chaque cluster
rfm_clust_df = model.transform(final_df_scaled)
rfm_clust_df.show(truncate=False)


+----------+-------------------+------------------+------------+------------------------------+--------------------------------------------------------------+----------+
|CustomerID|nbr_total_commandes|montant_total     |Recency_days|features                      |scaledFeatures                                                |prediction|
+----------+-------------------+------------------+------------+------------------------------+--------------------------------------------------------------+----------+
|12346     |2                  |0.0               |325         |[2.0,0.0,325.0]               |[0.21416132869209523,0.0,3.225097752344116]                   |2         |
|12347     |7                  |4310.000016212463 |2           |[7.0,4310.000016212463,2.0]   |[0.7495646504223333,0.5244140818236264,0.019846755399040714]  |0         |
|12348     |4                  |1797.2399997711182|75          |[4.0,1797.2399997711182,75.0] |[0.42832265738419045,0.2186770210560028,0.7442533274640

### 5) Analyser les clusters

In [23]:
# nombre de clients / cluster
group_cluster = rfm_clust_df.groupBy('prediction').count().orderBy('prediction')
group_cluster.show()

+----------+-----+
|prediction|count|
+----------+-----+
|         0| 2930|
|         1|   21|
|         2| 1071|
|         3|    4|
|         4|  346|
+----------+-----+



In [24]:
# moyenne rfm par cluster
mean_cluster = (
    rfm_clust_df
      .groupBy("prediction")
      .agg(
          F.count("*").alias("n_clients"),
          F.round(F.avg("Recency_days"), 2).alias("avg_Recency"),
          F.round(F.avg("nbr_total_commandes"), 2).alias("avg_Frequency"),
          F.round(F.avg("montant_total"), 2).alias("avg_Monetary"),
      )
      .orderBy("prediction")
)

mean_cluster.show()

+----------+---------+-----------+-------------+------------+
|prediction|n_clients|avg_Recency|avg_Frequency|avg_Monetary|
+----------+---------+-----------+-------------+------------+
|         0|     2930|      43.98|         3.81|     1154.25|
|         1|       21|       4.71|        91.33|    51270.45|
|         2|     1071|     249.04|          1.8|      452.05|
|         3|        4|       8.25|         55.0|   211743.78|
|         4|      346|      13.47|        20.11|     7192.92|
+----------+---------+-----------+-------------+------------+



Nous allons renommer nos clients comme:

*   0	Dormants
*   1	VIP
*   2	Top VIP
*   3	Actifs réguliers
*   4	Gros clients

In [25]:
rfm_clust_df = rfm_clust_df.withColumn(
    "type_client",
    F.when(F.col("prediction") == 0, "Dormants")
     .when(F.col("prediction") == 1, "VIP")
     .when(F.col("prediction") == 2, "Top VIP")
     .when(F.col("prediction") == 3, "Actifs réguliers")
     .when(F.col("prediction") == 4, "Gros clients")
     .otherwise("Other")
)
rfm_clust_df.show()

+----------+-------------------+------------------+------------+--------------------+--------------------+----------+------------+
|CustomerID|nbr_total_commandes|     montant_total|Recency_days|            features|      scaledFeatures|prediction| type_client|
+----------+-------------------+------------------+------------+--------------------+--------------------+----------+------------+
|     12346|                  2|               0.0|         325|     [2.0,0.0,325.0]|[0.21416132869209...|         2|     Top VIP|
|     12347|                  7| 4310.000016212463|           2|[7.0,4310.0000162...|[0.74956465042233...|         0|    Dormants|
|     12348|                  4|1797.2399997711182|          75|[4.0,1797.2399997...|[0.42832265738419...|         0|    Dormants|
|     12349|                  1|1757.5499963760376|          18|[1.0,1757.5499963...|[0.10708066434604...|         0|    Dormants|
|     12350|                  1|334.39999771118164|         310|[1.0,334.39999771..

In [26]:
# connaitre la médiane
median = rfm_clust_df.approxQuantile(
    ["Recency_days", "nbr_total_commandes", "montant_total"],
    [0.5],
    0.01
)
print(median)

[[50.0], [3.0], [643.6299734115601]]


## 4) Modélisation supervisé

*	Classification : prédire si un client est un “gros dépensier” (1) ou “petit
dépensier” (0), en définissant un seuil sur monetary_value.
    Ex. Label = 1 si monetary_value > 500, sinon 0.

*	Ou : prédire si un client revient dans le mois suivant (churn vs non-churn) en définissant un label binaire.

*	Ou : une régression pour estimer la monetary_value à partir de variables comme frequency, country, etc.

Tâches :

1.	Choisir la cible (label) et les features.
2.	Diviser le dataset en train / test (ex. 70% - 30%).
3.	Assembler les features dans un vecteur (VectorAssembler).
4.	Entraîner un ou plusieurs algorithmes (LogisticRegression, RandomForest, ou autre) sur le jeu d’entraînement.
5.	Évaluer les performances sur le jeu de test :
o	Pour la classification : accuracy, precision, recall, F1, matrice de confusion, etc.
o	Pour la régression : RMSE, MAE, R².
(Optionnel) Optimisation des hyperparamètres via CrossValidator ou TrainValidationSplit.

In [27]:
# Prédiction : on souhaite prédire si un client est un gros dépensier (1)
# ou petit dépensier (0) avec un seuil de 643 (afin d'avoir une bonne distribution des classes)

monetary_value = 643.63
df_supervised = df_rfm.withColumn(
    "label",
    F.when(F.col("montant_total") > monetary_value, 1).otherwise(0)
)

# Affichage de la distribution des clients
import plotly.express as px
counts = df_supervised.groupBy('label').count()
counts_pd = counts.toPandas()
fig = px.pie(
    counts_pd,
    values="count",
    names="label",
    title="Distribution des clients (0 for <643.63 ; 1 for > 643.63)."
)
fig.show()

In [28]:


features_columns = ['nbr_total_commandes', 'Recency_days']
train_data, test_data = df_supervised.randomSplit([0.7, 0.3], seed = 12)
# assemblage des features dans une liste pour chaque client
assembler = VectorAssembler(inputCols=features_columns, outputCol='features')
# rassemblement avec le df
train_data = assembler.transform(train_data)
test_data = assembler.transform(test_data)

# check avant entrainement
print("Train Dataset Count: " + str(train_data.count()))
print("Test Dataset Count: " + str(test_data.count()))
train_data.show(20, truncate=False)
test_data.show(20, truncate=False)

Train Dataset Count: 3090
Test Dataset Count: 1282
+----------+-------------------+------------------+------------+-----+-----------+
|CustomerID|nbr_total_commandes|montant_total     |Recency_days|label|features   |
+----------+-------------------+------------------+------------+-----+-----------+
|12346     |2                  |0.0               |325         |0    |[2.0,325.0]|
|12349     |1                  |1757.5499963760376|18          |1    |[1.0,18.0] |
|12350     |1                  |334.39999771118164|310         |0    |[1.0,310.0]|
|12352     |11                 |1545.4099912643433|36          |1    |[11.0,36.0]|
|12353     |1                  |89.0              |204         |0    |[1.0,204.0]|
|12355     |1                  |459.3999996185303 |214         |0    |[1.0,214.0]|
|12356     |3                  |2811.429984331131 |22          |1    |[3.0,22.0] |
|12357     |1                  |6207.669998168945 |33          |1    |[1.0,33.0] |
|12358     |2                  |1168

### Model implementation

In [29]:
# GBT Classification
gbtclassifier = GBTClassifier(featuresCol = 'features', labelCol = 'label', maxIter = 20)
model = gbtclassifier.fit(train_data)
model

GBTClassificationModel: uid = GBTClassifier_baad45668471, numTrees=20, numClasses=2, numFeatures=2

In [30]:
# Logistic regression
lr = LogisticRegression(featuresCol="features", labelCol="label")
model_rl = lr.fit(train_data)
model_rl

LogisticRegressionModel: uid=LogisticRegression_144cd91ef40f, numClasses=2, numFeatures=2

testons les modèles

In [31]:
# prediction de la GBT
gbt_predictions = model.transform(test_data)
gbt_predictions.select("features","prediction","label").show(20)

+-----------+----------+-----+
|   features|prediction|label|
+-----------+----------+-----+
|  [7.0,2.0]|       1.0|    1|
| [4.0,75.0]|       1.0|    1|
|[1.0,232.0]|       0.0|    1|
|[3.0,291.0]|       0.0|    0|
|  [1.0,4.0]|       0.0|    0|
| [1.0,25.0]|       0.0|    1|
|  [3.0,2.0]|       1.0|    0|
|[2.0,315.0]|       0.0|    1|
|[1.0,129.0]|       0.0|    1|
|[2.0,337.0]|       0.0|    0|
|[15.0,15.0]|       1.0|    1|
|[4.0,119.0]|       1.0|    1|
|[1.0,148.0]|       0.0|    1|
|[3.0,301.0]|       0.0|    1|
| [1.0,63.0]|       0.0|    0|
|[2.0,162.0]|       0.0|    1|
| [5.0,11.0]|       1.0|    1|
|  [7.0,0.0]|       1.0|    1|
|[1.0,366.0]|       0.0|    0|
| [1.0,22.0]|       0.0|    0|
+-----------+----------+-----+
only showing top 20 rows



In [32]:
# prediction de la regression
predictions = model_rl.transform(train_data)
predictions.select("features", "probability", "prediction", "label").show()

+-----------+--------------------+----------+-----+
|   features|         probability|prediction|label|
+-----------+--------------------+----------+-----+
|[2.0,325.0]|[0.88322236490133...|       0.0|    0|
| [1.0,18.0]|[0.83031637613792...|       0.0|    1|
|[1.0,310.0]|[0.94534940125507...|       0.0|    0|
|[11.0,36.0]|[7.05541642083718...|       1.0|    1|
|[1.0,204.0]|[0.91623123551549...|       0.0|    0|
|[1.0,214.0]|[0.91949109221990...|       0.0|    0|
| [3.0,22.0]|[0.45533410422778...|       1.0|    1|
| [1.0,33.0]|[0.83926062879380...|       0.0|    1|
|  [2.0,1.0]|[0.65071908044280...|       0.0|    1|
|  [6.0,7.0]|[0.05115079684703...|       1.0|    1|
| [3.0,52.0]|[0.48765019178339...|       1.0|    1|
|[1.0,287.0]|[0.93997732496188...|       0.0|    0|
| [13.0,3.0]|[1.02776212581596...|       1.0|    1|
|[2.0,109.0]|[0.74823785531848...|       0.0|    0|
|  [4.0,7.0]|[0.24302572447971...|       1.0|    1|
| [4.0,51.0]|[0.27971240917289...|       1.0|    1|
| [2.0,44.0]

### Model evaluation

In [33]:
# evaluation de la GBT
gbtclassifierevaluator = BinaryClassificationEvaluator()
ROC = gbtclassifierevaluator.evaluate(gbt_predictions, {gbtclassifierevaluator.metricName: "areaUnderROC"})
print("Test Area Under ROC: " , ROC)

precision_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
recall_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")

precision = precision_eval.evaluate(gbt_predictions)
recall = recall_eval.evaluate(gbt_predictions)
f1 = 2 * (precision * recall) / (precision + recall)

print("Precision: ", precision)
print("Recall: ", recall)
print("F1 score: ", f1)

Test Area Under ROC:  0.8933973984215143
Precision:  0.8231722076389536
Recall:  0.8229329173166926
F1 score:  0.8230525450852952


In [34]:
# evaluation de la regression logistic par RMSE et R2
evaluator_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)

evaluator_r2 = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)

print("RMSE:", rmse)
print("R2:", r2)

RMSE: 0.40983065411038216
R2: 0.32807398891205386


L’arbre de décision en classification présente de bonnes performances prédictives avec une AUC de 0,89, une précision et un rappel proches de 0,82 ainsi qu’un score F1 de 0,82, ce qui indique une bonne capacité de séparation des classes et un équilibre entre faux positifs et faux négatifs. À l’inverse, le modèle de régression montre un pouvoir explicatif limité, avec un R² de 0,27 et un RMSE de 0,42, suggérant qu’il n’explique qu’une part modeste de la variance. Globalement, le modèle de classification apparaît donc plus adapté pour identifier les clients à forte dépense.

### 5) Visualisations & Recommandations

- **Illustration des résultats**  
  Visualisation des performances du clustering et de la classification à l’aide de graphiques :
  - Histogrammes des distributions (Recency, Frequency, Monetary)
  - Barplots du nombre de clients par cluster
  - Comparaison des métriques (accuracy, precision, recall, etc.)

- **Proposition d’actions ou recommandations supplémentaires**
  - Cibler un cluster spécifique pour des offres marketing personnalisées.
  - Définir un plan de conversion des “petits dépensiers” vers des “gros dépensiers”.
  - Expliquer comment intégrer le modèle supervisé dans un pipeline de recommandation (scoring client en temps réel, segmentation dynamique, etc.).


In [35]:
# histogrammes des distributions des variables R, F et M
# color_map afin de conserver le même code couleur pour les différents graphiques
color_map = {
    "Actifs réguliers": "#1A689F",
    "Dormants": "#ff7f0e",
    "Top VIP": "#2ca02c",
    "Gros clients": "#d62728",
    "VIP": "#9467bd",
    "Other": "#8c564b"
}

# convertion du df en pandas pour affichage
rfm_sample = (
    rfm_clust_df
        .select("type_client", "Recency_days", "nbr_total_commandes", "montant_total")
        .toPandas()
)

px.histogram(
    rfm_sample, x="Recency_days", color="type_client", nbins=60,
    color_discrete_map=color_map,
    title="Distribution de Recency (jours) par cluster"
).show()

px.histogram(
    rfm_sample, x="nbr_total_commandes", color="type_client", nbins=60,
    color_discrete_map=color_map,
    title="Distribution de Frequency (nombre de commandes) par cluster"
).show()

px.histogram(
    rfm_sample, x="montant_total", color="type_client", nbins=60,
    color_discrete_map=color_map,
    title="Distribution de Monetary (montant total) par cluster"
).show()

In [36]:
# répartition des clients par cluster (pie)
cluster_counts = rfm_clust_df.groupBy("type_client").count().toPandas()
fig = px.pie(
    cluster_counts, values="count", names="type_client", color="type_client",
    color_discrete_map=color_map,
    title="Répartition des clients par cluster"
)
fig.show()

In [37]:
# comparaison des métriques
metrics = {
    "Precision": precision,
    "Recall": recall,
    "F1 score" : f1,
    "ROC" :  ROC,
    "RMSE": rmse,
    "R2" : r2
}
fig = px.bar(
    x=list(metrics.keys()),
    y=list(metrics.values()),
    range_y=[0, 1],
    title="Comparaison des métriques"
)
fig.show()

### La segmentation RFM permet d’identifier des profils clients distincts et exploitables opérationnellement.

**VIP / Top VIP**

* Programme fidélité personnalisé
* Objectif : maximiser la Customer Lifetime Value et renforcer la fidélisation.

**Actifs réguliers**

* Promotions à seuil (ex : remise au-delà de 50£)
* Packs produits complémentaires

Objectif : augmenter le panier moyen et la fréquence d’achat.

**Dormants**

* Campagnes de réactivation ciblées
* Email personnalisé basé sur le dernier achat
* coupon retour

Objectif : réduire le churn et réactiver la base client.


### Conversion des “petits dépensiers” vers “gros dépensiers”

Utilisation du modèle de classification pour prédire la probabilité qu’un client devienne un “gros dépensier”. Seuls les clients à fort potentiel sont ciblés.

* Recommandations produits premium
* Bundles inspirés des paniers des gros clients
* Incitation à l’achat récurrent

* Suivi de l’évolution des indicateurs RFM

Objectif : optimiser l’allocation budgétaire marketing.

### Intégration du modèle dans un pipeline de recommandation Pipeline batch

Mise à jour quotidienne des transactions
Recalcul des variables RFM
Export des segments et scores vers le CRM
Segmentation dynamique

* **À chaque nouvelle transaction :**

  * Mise à jour des indicateurs client
  * Recalcul du score
  * Adaptation automatique du segment

  * Le client peut évoluer d’un cluster à un autre en fonction de son comportement.

* **Permet de proposer**

  * Recommandations personnalisées
  * Offres ciblées selon la probabilité de montée en gamme